In [1]:
import os
from copy import deepcopy
from typing import (
    Any,
    AsyncIterable,
    Callable,
    Dict,
    Generator,
    List,
    NamedTuple,
    Optional,
    Tuple,
    Union,
)
import requests
from io import BytesIO

from PIL import Image
import torch
from accelerate import infer_auto_device_map, load_checkpoint_and_dispatch, init_empty_weights

from data.transforms import ImageTransform
from data.data_utils import pil_img2rgb, add_special_tokens
from modeling.bagel import (
    BagelConfig, Bagel, Qwen2Config, Qwen2ForCausalLM, SiglipVisionConfig, SiglipVisionModel
)
from modeling.qwen2 import Qwen2Tokenizer
from modeling.bagel.qwen2_navit import NaiveCache
from modeling.autoencoder import load_ae
from safetensors.torch import load_file

# model_path = "/path/to/BAGEL-7B-MoT/weights"  # Download from https://huggingface.co/ByteDance-Seed/BAGEL-7B-MoT

model_path = "/mnt/data1/jiwon/BAGEL/models/BAGEL-7B-MoT"

# LLM config preparing
llm_config = Qwen2Config.from_json_file(os.path.join(model_path, "llm_config.json"))
llm_config.qk_norm = True
llm_config.tie_word_embeddings = False
llm_config.layer_module = "Qwen2MoTDecoderLayer"

# ViT config preparing
vit_config = SiglipVisionConfig.from_json_file(os.path.join(model_path, "vit_config.json"))
vit_config.rope = False
vit_config.num_hidden_layers = vit_config.num_hidden_layers - 1

# VAE loading
vae_model, vae_config = load_ae(local_path=os.path.join(model_path, "ae.safetensors"))
vae_model = vae_model.to(torch.bfloat16)

# Bagel config preparing
config = BagelConfig(
    visual_gen=True,
    visual_und=True,
    llm_config=llm_config, 
    vit_config=vit_config,
    vae_config=vae_config,
    vit_max_num_patch_per_side=70,
    connector_act='gelu_pytorch_tanh',
    latent_patch_size=2,
    max_latent_size=64,
)

# with init_empty_weights():
language_model = Qwen2ForCausalLM(llm_config).to(torch.bfloat16)
vit_model      = SiglipVisionModel(vit_config).to(torch.bfloat16)
model          = Bagel(language_model, vit_model, config).to(torch.bfloat16)
model.vit_model.vision_model.embeddings.convert_conv2d_to_linear(vit_config, meta=False)


# Tokenizer Preparing
tokenizer = Qwen2Tokenizer.from_pretrained(model_path)
tokenizer, new_token_ids, _ = add_special_tokens(tokenizer)

# Image Transform Preparing
vae_transform = ImageTransform(1024, 512, 16)
vit_transform = ImageTransform(980, 224, 14)

# device_map = 'cuda:6'
# print(device_map)

# same_device_modules = [
#     'language_model.model.embed_tokens',
#     'time_embedder',
#     'latent_pos_embed',
#     'vae2llm',
#     'llm2vae',
#     'connector',
#     'vit_pos_embed'
# ]


# # Thanks @onion-liu: https://github.com/ByteDance-Seed/Bagel/pull/8
# model = load_checkpoint_and_dispatch(
#     model,
#     checkpoint=os.path.join(model_path, "ema.safetensors"),
#     offload_buffers=False,
#     dtype=torch.bfloat16,
# )

model = model.eval()
print('Model loaded')

Model loaded


In [ ]:
device = torch.device('cuda:6' if torch.cuda.is_available() else 'cpu')
model.to(device)
[vae_model.to(device)

AutoEncoder(
  (encoder): Encoder(
    (conv_in): Conv2d(3, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (down): ModuleList(
      (0): Module(
        (block): ModuleList(
          (0-1): 2 x ResnetBlock(
            (norm1): GroupNorm(32, 128, eps=1e-06, affine=True)
            (conv1): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (norm2): GroupNorm(32, 128, eps=1e-06, affine=True)
            (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          )
        )
        (attn): ModuleList()
        (downsample): Downsample(
          (conv): Conv2d(128, 128, kernel_size=(3, 3), stride=(2, 2))
        )
      )
      (1): Module(
        (block): ModuleList(
          (0): ResnetBlock(
            (norm1): GroupNorm(32, 128, eps=1e-06, affine=True)
            (conv1): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (norm2): GroupNorm(32, 256, eps=1e-06, affine=True)
      

In [ ]:
model

Bagel(
  (language_model): Qwen2ForCausalLM(
    (model): Qwen2Model(
      (embed_tokens): Embedding(152064, 3584)
      (layers): ModuleList(
        (0-27): 28 x Qwen2MoTDecoderLayer(
          (self_attn): PackedAttentionMoT(
            (q_proj): Linear(in_features=3584, out_features=3584, bias=True)
            (k_proj): Linear(in_features=3584, out_features=512, bias=True)
            (v_proj): Linear(in_features=3584, out_features=512, bias=True)
            (o_proj): Linear(in_features=3584, out_features=3584, bias=False)
            (q_norm): Qwen2RMSNorm((128,), eps=1e-06)
            (k_norm): Qwen2RMSNorm((128,), eps=1e-06)
            (q_norm_moe_gen): Qwen2RMSNorm((128,), eps=1e-06)
            (k_norm_moe_gen): Qwen2RMSNorm((128,), eps=1e-06)
            (q_proj_moe_gen): Linear(in_features=3584, out_features=3584, bias=True)
            (k_proj_moe_gen): Linear(in_features=3584, out_features=512, bias=True)
            (v_proj_moe_gen): Linear(in_features=3584, out_fea